# 🚀 YOLO26n Training - Phát hiện U xơ (Myoma)

Notebook này được thiết kế để huấn luyện mô hình **YOLO26n** trên **Google Colab** (GPU miễn phí T4).

## Quy trình:
1. Cài đặt Ultralytics
2. Mount Google Drive & giải nén dataset
3. Kiểm tra GPU
4. Huấn luyện mô hình
5. Đánh giá và xuất kết quả

> **Lưu ý:** Trước khi chạy, hãy vào **Runtime > Change runtime type > GPU (T4)** để bật GPU.

## 1️⃣ Cài đặt Ultralytics (YOLO26n)

In [ ]:
!pip install -q ultralytics>=8.3

import ultralytics
ultralytics.checks()
print(f"\n\u2705 Ultralytics version: {ultralytics.__version__}")

## 2️⃣ Mount Google Drive & Giải nén Dataset

**Chuẩn bị trước:**
- Chạy `convert_voc_to_yolo_v2.py` trên máy cá nhân để tạo thư mục `datasets_v2/`
- Nén thư mục `datasets_v2/` thành `datasets_v2.zip`
- Tải `datasets_v2.zip` và `dataset_v2.yaml` lên Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

# ============================================================
# CẤU HÌNH - Chỉnh sửa đường dẫn cho phù hợp với Google Drive của bạn
# ============================================================
DRIVE_DATASET_ZIP = "/content/drive/MyDrive/datasets_v2.zip"
DRIVE_DATASET_YAML = "/content/drive/MyDrive/dataset_v2.yaml"
WORK_DIR = "/content/yolo26n_myoma"
DRIVE_RUNS_DIR = "/content/drive/MyDrive/yolo26n_myoma/runs"
# ============================================================

os.makedirs(WORK_DIR, exist_ok=True)
%cd {WORK_DIR}
print(f"\u2705 Working directory: {os.getcwd()}")

# Giải nén dataset
if os.path.exists(DRIVE_DATASET_ZIP):
    !unzip -q -o "{DRIVE_DATASET_ZIP}" -d "{WORK_DIR}"
    print("\u2705 Dataset đã được giải nén!")
else:
    print(f"\u274c Không tìm thấy: {DRIVE_DATASET_ZIP}")
    print("Hãy tải datasets_v2.zip lên Google Drive trước!")

# Copy dataset_v2.yaml
if os.path.exists(DRIVE_DATASET_YAML):
    !cp "{DRIVE_DATASET_YAML}" "{WORK_DIR}/dataset_v2.yaml"
    print("\u2705 dataset_v2.yaml đã được copy!")
else:
    print(f"\u274c Không tìm thấy: {DRIVE_DATASET_YAML}")
    print("Hãy tải dataset_v2.yaml lên Google Drive trước!")

In [ ]:
# Tạo dataset_v2.yaml với absolute path để tránh lỗi file not found
import yaml
yaml_content = {
    'path': os.path.join(WORK_DIR, 'datasets_v2'),
    'train': 'images/train',
    'val': 'images/val',
    'names': {0: 'Myoma'}
}
with open(os.path.join(WORK_DIR, 'dataset_v2.yaml'), 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)
print("✅ Đã tạo dataset_v2.yaml bằng absolute path!")


## 3️⃣ Kiểm tra GPU

In [ ]:
print('Đã bỏ qua kiểm tra GPU theo yêu cầu.')


## 4️⃣ Huấn luyện YOLO26n

### Các tham số chính:
| Tham số | Giá trị | Giải thích |
|---------|---------|----------|
| `model` | `yolo26n.pt` | Model nhỏ (nano) - phù hợp Colab free |
| `epochs` | 150 | Số epoch train |
| `imgsz` | 640 | Kích thước ảnh đầu vào |
| `batch` | -1 | Tự động chọn batch size tối ưu cho GPU |
| `patience` | 30 | Early stopping nếu không cải thiện sau 30 epoch |

> **Tip:** Nếu bạn muốn model chính xác hơn, thay `yolo26n.pt` bằng `yolo26s.pt` (small) nhưng sẽ chậm hơn và tốn nhiều VRAM hơn.

In [ ]:
# CẤU HÌNH TRAIN HARD AUGMENTATION (CHỐNG LỆCH PHA VÀ ẢO GIÁC)
from ultralytics import YOLO

# Khởi tạo mô hình gốc chuẩn (KHÔNG dùng best.pt cũ)
model = YOLO('yolo26n.pt') 

# Bắt đầu quá trình huấn luyện 'Kỷ luật thép'
results = model.train(
    data="dataset_v2.yaml",     # File cấu hình dataset
    epochs=150,               # Số epoch
    imgsz=640,                # Kích thước ảnh
    batch=-1,                 # Auto batch size (tối ưu cho GPU)
    patience=30,              # Early stopping
    save=True,                # Lưu checkpoint
    save_period=10,           # Lưu mỗi 10 epoch
    device=0,                 # GPU 0
    workers=2,                # Số worker load data
    project="/content/drive/MyDrive/yolo26n_myoma/runs", # Lưu lên Drive
    name="myoma_robust_v2",   # Tên phiên bản mới
    exist_ok=True,            # Ghi đè nếu đã tồn tại
    pretrained=True,          # Sử dụng pretrained weights
    optimizer="auto",         # Tự chọn optimizer
    lr0=0.01,                 # Learning rate ban đầu
    lrf=0.01,                 # Learning rate cuối
    warmup_epochs=3,          # Số epoch warmup
    cos_lr=True,              # Cosine LR scheduler
    hsv_h=0.015,              # Augment: Hue
    hsv_s=0.5,                # Giữ nguyên độ bão hòa
    hsv_v=0.4,                # Giữ nguyên độ sáng tối
    degrees=10.0,             # Xoay ảnh
    translate=0.2,            # Xê dịch khối u
    flipud=0.5,               # Flip dọc (hữu ích cho ảnh siêu âm)
    fliplr=0.5,               # Flip ngang
    scale=0.5,                # TĂNG LÊN 0.5: Phóng to/thu nhỏ mạnh (Chống viền đen)
    mosaic=1.0,               # BẬT 100%: Ghép 4 ảnh trị lỗi Scale
    erasing=0.4,              # THÊM MỚI: Xóa ngẫu nhiên 40% chống Marker
)

## 5️⃣ Đánh giá mô hình

In [ ]:
# Đánh giá trên tập validation
metrics = model.val()

print("\n" + "=" * 50)
print("  KẾT QUẢ ĐÁNH GIÁ")
print("=" * 50)
print(f"  mAP50      : {metrics.box.map50:.4f}")
print(f"  mAP50-95   : {metrics.box.map:.4f}")
print(f"  Precision  : {metrics.box.mp:.4f}")
print(f"  Recall     : {metrics.box.mr:.4f}")
print("=" * 50)

In [ ]:
# Hiển thị biểu đồ training
from IPython.display import Image, display

result_dir = os.path.join(DRIVE_RUNS_DIR, "myoma_yolo26n")

# Confusion matrix
print("Confusion Matrix:")
display(Image(filename=f"{result_dir}/confusion_matrix.png", width=600))

# Training results
print("\nTraining Curves:")
display(Image(filename=f"{result_dir}/results.png", width=800))

# Sample predictions
print("\nSample Predictions (Val):")
display(Image(filename=f"{result_dir}/val_batch0_pred.jpg", width=800))

## 6️⃣ Xuất mô hình & Lưu về Drive

In [ ]:
import os
import shutil
from ultralytics import YOLO

# 1. Xác định đường dẫn file best.pt
DRIVE_RUNS_DIR = "/content/drive/MyDrive/yolo26n_myoma/runs"
result_dir = os.path.join(DRIVE_RUNS_DIR, "myoma_yolo26n")
best_pt = f"{result_dir}/weights/best.pt"

# 2. Load lại model tốt nhất và xuất ONNX
print("Xuất mô hình sang ONNX...")
best_model = YOLO(best_pt)
best_model.export(format="onnx", imgsz=640, simplify=True)

# 3. Lưu best weights về Google Drive (thư mục ngoài)
DRIVE_SAVE_DIR = "/content/drive/MyDrive/yolo26n_myoma_results"
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)

# Copy best model
if os.path.exists(best_pt):
    shutil.copy2(best_pt, f"{DRIVE_SAVE_DIR}/best.pt")
    print(f"Best model đã lưu: {DRIVE_SAVE_DIR}/best.pt")

# Copy ONNX model
best_onnx = f"{result_dir}/weights/best.onnx"
if os.path.exists(best_onnx):
    shutil.copy2(best_onnx, f"{DRIVE_SAVE_DIR}/best.onnx")
    print(f"ONNX model đã lưu: {DRIVE_SAVE_DIR}/best.onnx")

# Copy kết quả training
for fname in ["results.png", "confusion_matrix.png", "results.csv"]:
    src = f"{result_dir}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, f"{DRIVE_SAVE_DIR}/{fname}")

print(f"\nTất cả kết quả đã được lưu vào: {DRIVE_SAVE_DIR}")


## 7️⃣ Thử nghiệm dự đoán trên ảnh mới (tuỳ chọn)

In [ ]:
from ultralytics import YOLO
from IPython.display import Image, display

# Load best model từ Google Drive
best_model = YOLO(os.path.join(DRIVE_RUNS_DIR, "myoma_yolo26n/weights/best.pt"))

# Chọn 1 ảnh từ tập val để thử
import glob
val_images = sorted(glob.glob("datasets_v2/images/val/*.jpg"))

if val_images:
    test_img = val_images[0]
    print(f"Dự đoán trên: {test_img}")
    
    results = best_model.predict(
        source=test_img,
        save=True,
        conf=0.25,
        project="runs/predict",
        name="test",
        exist_ok=True,
    )
    
    # Hiển thị kết quả
    import os
    pred_img = os.path.join("runs/predict/test", os.path.basename(test_img))
    if os.path.exists(pred_img):
        display(Image(filename=pred_img, width=640))
    
    # In chi tiết
    for r in results:
        print(f"\n\u0110ối tượng phát hiện: {len(r.boxes)}")
        for box in r.boxes:
            cls_id = int(box.cls)
            conf = float(box.conf)
            xyxy = box.xyxy[0].tolist()
            print(f"  - Class: {r.names[cls_id]} | Conf: {conf:.3f} | Box: {xyxy}")
else:
    print("Không tìm thấy ảnh trong tập val")

### LƯU Ý QUAN TRỌNG KHI INFERENCE TRÊN ẢNH MỚI (DEPLOY)
Do mô hình đã được huấn luyện trên tập dữ liệu đã qua xử lý (Crop + Grayscale), khi bạn dự đoán trên một bức ảnh siêu âm MỚI HOÀN TOÀN (chưa qua xử lý), bạn **BẮT BUỘC** phải tiền xử lý nó trước khi đưa vào `best_model.predict()`.

**Code tiền xử lý mẫu:**
```python
import cv2
from PIL import Image

def preprocess_for_inference(img_path):
    img = Image.open(img_path).convert("RGB")
    # 1. Tự động crop viền (ví dụ cắt 15% top, 5% bottom)
    w, h = img.size
    img = img.crop((int(w*0.05), int(h*0.15), int(w*0.90), int(h*0.95)))
    # 2. Chuyển sang Grayscale (Thanh tẩy màu)
    import numpy as np
    cv_img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2BGR)
    cv_gray = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
    cv_gray_3ch = cv2.cvtColor(cv_gray, cv2.COLOR_GRAY2RGB)
    return Image.fromarray(cv_gray_3ch)
```

In [ ]:
# CHẠY DỰ ĐOÁN VÀ LƯU CHI TIẾT TỪNG ẢNH VAL (Lưu tọa độ & Confidence vào file txt)
from ultralytics import YOLO
import glob

# Load mô hình tốt nhất vừa train xong (Thường nằm ở runs/detect/train/weights/best.pt)
# Nếu bạn đặt tên khác ở bước Train, hãy sửa lại đường dẫn này nhé!
weight_path = 'runs/detect/train/weights/best.pt'
model = YOLO(weight_path)

# Chạy inference trên tập validation và yêu cầu xuất ra file txt chi tiết
print("Đang phân tích chi tiết từng ảnh Validation...")
results = model.predict(
    source='datasets_v2/images/val', 
    save=True,          # Lưu lại ảnh có vẽ Bounding Box
    save_txt=True,      # BẮT BUỘC: Lưu chi tiết tọa độ ra file txt
    save_conf=True,     # BẮT BUỘC: Ghi kèm độ tự tin (Confidence)
    name='val_detailed_results'
)

print("\n✅ Phân tích hoàn tất!")
print("Bạn hãy mở thư mục ở menu bên trái Colab: runs/detect/val_detailed_results/labels/")
print("Tại đây, mỗi bức ảnh sẽ có 1 file .txt lưu trữ tọa độ và confidence chi tiết!")